<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/inspect-adapters.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
# !pip -qqq install peft trl

In [ ]:
# !unzip lora-adapter_1750_1760.zip -d .

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import defaultdict
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")

In [ ]:
def compare_peft_lora_adapters(
    base_model_name_or_path: str,
    adapter_path_1: str,
    adapter_path_2: str,
    device: str = "cpu",
    atol: float = 1e-6,
):
    """
    Compare two PEFT LoRA adapters trained on the same base model.

    Args:
        base_model_name_or_path: HF model id or local path
        adapter_path_1: path to first adapter
        adapter_path_2: path to second adapter
        device: cpu or cuda
        atol: tolerance for equality check

    Returns:
        dict with per-tensor difference statistics
    """

    # Load base model once
    base_model_1 = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
    ).to(device)

    base_model_2 = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
    ).to(device)

    # Load adapters separately
    model1 = PeftModel.from_pretrained(base_model_1, adapter_path_1).to(device)
    model2 = PeftModel.from_pretrained(base_model_2, adapter_path_2).to(device)

    # Extract only LoRA weights
    sd1 = {
        k: v for k, v in model1.state_dict().items()
        if "lora_" in k
    }
    sd2 = {
        k: v for k, v in model2.state_dict().items()
        if "lora_" in k
    }

    report = {}
    changed = []

    for key in sd1.keys():
        if key not in sd2:
            print(f"[Missing in adapter2] {key}")
            continue

        w1 = sd1[key]
        w2 = sd2[key]

        diff = w1 - w2

        l2 = torch.norm(diff).item()
        max_abs = diff.abs().max().item()
        rel = l2 / (torch.norm(w1).item() + 1e-12)

        cos_dist = 1.0 -  torch.nn.functional.cosine_similarity(
            w1.flatten(), w2.flatten(), dim=0
                ).item()

        equal = torch.allclose(w1, w2, atol=atol)

        report[key] = {
            "equal": equal,
            "l2_diff": l2,
            "max_abs_diff": max_abs,
            "relative_l2": rel,
            'cosine': cos_dist,
        }

        if not equal:
            changed.append(key)

    print(f"\nTotal LoRA tensors: {len(sd1)}")
    print(f"Changed tensors: {len(changed)}")

    return report


def plot_lora_difference_heatmap(report, stat='cosine'):
    """
    Creates a heatmap of LoRA L2 differences aggregated by:
    Transformer layer × projection module (q_proj, k_proj, etc.)

    Args:
        report: output dict from compare_peft_lora_adapters()
    """

    # Aggregate differences
    layer_module_diff = defaultdict(lambda: defaultdict(float))
    modules = set()
    layers = set()

    for key, stats in report.items():
        if stats["l2_diff"] == 0:
            continue

        parts = key.split(".")

        # Extract layer number
        if "layers" in parts:
            layer_idx = int(parts[parts.index("layers") + 1])
        else:
            continue

        # Extract projection/module name
        # e.g. q_proj, k_proj, v_proj, o_proj, gate_proj, etc.
        module_name = None
        for part in parts:
            if part.endswith("_proj"):
                module_name = part
                break

        if module_name is None:
            continue

        layer_module_diff[layer_idx][module_name] += stats[stat]
        modules.add(module_name)
        layers.add(layer_idx)

    layers = sorted(layers)
    modules = sorted(modules)

    # Build matrix
    heatmap = np.zeros((len(layers), len(modules)))

    for i, layer in enumerate(layers):
        for j, module in enumerate(modules):
            heatmap[i, j] = layer_module_diff[layer].get(module, 0.0)

    # Single plot only
    plt.figure()
    plt.imshow(heatmap)
    plt.xticks(range(len(modules)), modules, rotation=45)
    plt.yticks(range(len(layers)), layers)
    plt.xlabel("Projection Module")
    plt.ylabel("Transformer Layer")
    plt.title(f"LoRA {stat.upper()} Heatmap")
    plt.colorbar()
    plt.tight_layout()
    plt.show()



In [ ]:
base_model_name_or_path = 'meta-llama/Meta-Llama-3-8B'
adapter_path_1 = './lora-adapter_1750_1760/checkpoint-1000'
adapter_path_2 = './lora-adapter_1750_1760/checkpoint-5393'

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:

report = compare_peft_lora_adapters(base_model_name_or_path,adapter_path_1,adapter_path_2)
plot_lora_difference_heatmap(report, 'cosine')

# LoRA Layer Influence

In [ ]:
base_model_name_or_path = 'meta-llama/Meta-Llama-3-8B'
adapter_path = './lora-adapter_1750_1760/checkpoint-5393'

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path)


# Load adapters separately
model = PeftModel.from_pretrained(base_model, adapter_path).to(device)
model.eval()
   

In [ ]:
prompt = "The capital of Russia is the city of"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

In [ ]:
import torch
def compute_lora_contribution(model, inputs):

    contributions = {}

    with torch.no_grad():
        baseline = model(**inputs).logits[:, -1, :]

    for name, module in model.named_modules():

        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):

            # Save original weights
            # old_A = module.lora_A.weight.data.clone()
            # old_B = module.lora_B.weight.data.clone()

            old_A = module.lora_A['default'].weight.data.clone()
            old_B = module.lora_B['default'].weight.data.clone()

            # Disable LoRA
            module.lora_A['default'].weight.data.zero_()
            module.lora_B['default'].weight.data.zero_()

            with torch.no_grad():
                logits = model(**inputs).logits[:, -1, :]

            # Measure change
            diff = torch.norm(baseline - logits).item()

            contributions[name] = diff

            # Restore weights
            module.lora_A['default'].weight.data.copy_(old_A)
            module.lora_B['default'].weight.data.copy_(old_B)

    return contributions

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    base_logits = outputs.logits[:, -1, :]

In [ ]:
pred_token_id = torch.argmax(base_logits, dim=-1)
pred_token = tokenizer.decode(pred_token_id)

print("Predicted next token:", pred_token)

In [ ]:
contribs = compute_lora_contribution(model, inputs)

for k, v in sorted(contribs.items(), key=lambda x: -x[1])[:10]:
    print(k, v)

In [ ]:
import torch


def compute_lora_contribution(model, tokenizer, sentence, device="cpu"):
    """
    Measure how much each LoRA module contributes to predicting
    the correct next token.

    Returns:
        dict {module_name: contribution_score}
    """

    model.eval()

    # tokenize
    inputs = tokenizer(sentence, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)

    # context (everything except last token)
    context_ids = input_ids[:, :-1]

    # true next token
    target_token = input_ids[:, -1]

    with torch.no_grad():
        outputs = model(input_ids=context_ids)
        logits = outputs.logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)

    baseline_prob = probs[0, target_token].item()

    contributions = {}

    for name, module in model.named_modules():

        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):

            # save weights
            old_A = module.lora_A['default'].weight.data.clone()
            old_B = module.lora_B['default'].weight.data.clone()

            # disable LoRA
            module.lora_A['default'].weight.data.zero_()
            module.lora_B['default'].weight.data.zero_()

            with torch.no_grad():
                outputs = model(input_ids=context_ids)
                logits = outputs.logits[:, -1, :]
                probs = torch.softmax(logits, dim=-1)

            prob_without = probs[0, target_token].item()

            # contribution = drop in probability
            contribution = baseline_prob - prob_without

            contributions[name] = contribution

            # restore LoRA
            module.lora_A['default'].weight.data.copy_(old_A)
            module.lora_B['default'].weight.data.copy_(old_B)

    # sort largest contribution first
    contributions = dict(
        sorted(contributions.items(), key=lambda x: -x[1])
    )

    return contributions

In [ ]:
sentence = "The capital of France is Paris"

scores = compute_lora_contribution(
    model,
    tokenizer,
    sentence,
    device
)

for layer, score in list(scores.items())[:10]:
    print(layer, score)

# Perplexity

In [ ]:
import torch

def compute_perplexity(model, tokenizer, sentence, device="cpu"):
    """
    Compute perplexity of a single sentence.

    Args:
        model: HuggingFace causal LM (optionally with LoRA)
        tokenizer: corresponding tokenizer
        sentence: input string
        device: cpu or cuda

    Returns:
        perplexity (float)
    """

    model.eval()

    encodings = tokenizer(sentence, return_tensors="pt")
    input_ids = encodings["input_ids"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss

    perplexity = torch.exp(loss).item()

    return perplexity

In [ ]:
base_model_perpl = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
).to(device)

In [ ]:
compute_perplexity(model, tokenizer, sentence, device)

In [ ]:
compute_perplexity(base_model_perpl, tokenizer, sentence, device)